In [1]:
#################################### IMPORT PACKAGES ####################################

import torch 
import torch.nn as nn 
import torch.optim as optim 

from PIL import Image, ImageFile 
ImageFile.LOAD_TRUNCATED_IMAGES = True

import cv2 

import os 
import numpy as np 
import pandas as pd 

import matplotlib.pyplot as plt 
import matplotlib.patches as patches 

from tqdm import tqdm

import torchvision.transforms.functional as F
from torchvision import transforms

In [2]:
################################## DEFINE SOME CONSTANTS ##################################
# Device 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load and save model variable 
LOAD_MODEL = False
SAVE_MODEL = True

# model checkpoint file name 
CHECKPOINT_FILE = "checkpoint.pth.tar"

# Anchor boxes scaled between 0 and 1 
# list de 9 tuples
ANCHORS = [                                                               
	(0.14105, 0.10625), (0.11015, 0.16249), (0.09875, 0.14781),
	(0.08998, 0.13511), (0.08254, 0.12333), (0.07731, 0.19987),
	(0.07835, 0.09580), (0.06312, 0.11607), (0.06459, 0.21892)]


# Batch size for training 
BATCH_SIZE = 1

# Learning rate for training 
learning_rate = 1e-4

# Number of epochs for training 
epochs = 50

# Image sizes
ORIG_IMG_WIDTH = 6000
ORIG_IMG_HEIGHT = 4000
RESIZED_IMG_WIDTH = 576
RESIZED_IMG_HEIGHT = 384
GRID_WIDTH = 576 // 32
GRID_HEIGHT = 384 // 32

# Scaling the anchors 
scaled_anchors = torch.tensor(ANCHORS) * torch.tensor([GRID_WIDTH, GRID_HEIGHT])
print("Scaled anchors: ", scaled_anchors)
scaled_anchors = scaled_anchors.to(DEVICE)  # si tu es sur GPU

# Class labels 
class_labels = [ 
	"Jelly_White","Jelly_Milk","Jelly_Black","Amandina","Creme_brulee",
	"Triangolo","Tentation_noir","Comtesse","Noblesse","Noir_authentique",
	"Passion_au_lait","Arabia","Stracciatella"
]


# Setting number of classes 
NUM_CLASSES = 13

Scaled anchors:  tensor([[2.5389, 1.2750],
        [1.9827, 1.9499],
        [1.7775, 1.7737],
        [1.6196, 1.6213],
        [1.4857, 1.4800],
        [1.3916, 2.3984],
        [1.4103, 1.1496],
        [1.1362, 1.3928],
        [1.1626, 2.6270]])


In [3]:
############################ DEFINE SOME HELPER FUNCTIONS ############################

# Defining a function to calculate Intersection over Union (IoU) 
def iou(box1, box2, is_pred=True):
	if is_pred: 
		# IoU score for prediction and label 
		# box1 (prediction) and box2 (label) are both in [x, y, width, height] format 
		
		# Box coordinates of prediction 
		b1_x1 = box1[..., 0:1] - box1[..., 2:3] / 2
		b1_y1 = box1[..., 1:2] - box1[..., 3:4] / 2
		b1_x2 = box1[..., 0:1] + box1[..., 2:3] / 2
		b1_y2 = box1[..., 1:2] + box1[..., 3:4] / 2

		# Box coordinates of ground truth 
		b2_x1 = box2[..., 0:1] - box2[..., 2:3] / 2
		b2_y1 = box2[..., 1:2] - box2[..., 3:4] / 2
		b2_x2 = box2[..., 0:1] + box2[..., 2:3] / 2
		b2_y2 = box2[..., 1:2] + box2[..., 3:4] / 2

		# Get the coordinates of the intersection rectangle 
		x1 = torch.max(b1_x1, b2_x1) 
		y1 = torch.max(b1_y1, b2_y1) 
		x2 = torch.min(b1_x2, b2_x2) 
		y2 = torch.min(b1_y2, b2_y2) 
		# Make sure the intersection is at least 0 
		intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0) 

		# Calculate the union area 
		box1_area = abs((b1_x2 - b1_x1) * (b1_y2 - b1_y1)) 
		box2_area = abs((b2_x2 - b2_x1) * (b2_y2 - b2_y1)) 
		union = box1_area + box2_area - intersection 

		# Calculate the IoU score 
		epsilon = 1e-6
		iou_score = intersection / (union + epsilon) 

		# Return IoU score 
		return iou_score 
	
	else: # this case is used in the dataset class to calculate IoU of anchor boxes with ground truth boxes (so don't need x, y coordinates), on veut juste la ressemblance de la GT avec les 9 ancres
		# IoU score based on width and height of bounding boxes 
		
		# Calculate intersection area 
		intersection_area = torch.min(box1[..., 0], box2[..., 0]) * torch.min(box1[..., 1], box2[..., 1]) 

		# Calculate union area 
		box1_area = box1[..., 0] * box1[..., 1] 
		box2_area = box2[..., 0] * box2[..., 1] 
		union_area = box1_area + box2_area - intersection_area 

		# Calculate IoU score 
		iou_score = intersection_area / union_area 

		# Return IoU score 
		return iou_score

##################################### CONVERTS CELLS TO BOUNDING BOXES #####################################

# anchors sera les scaled_anchors entre 0 et 26 et prediction est la sortie du modèle
def convert_cell_to_bboxes(prediction, anchors, grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT, resized_img_width=RESIZED_IMG_WIDTH, resized_img_height=RESIZED_IMG_HEIGHT, num_classes=NUM_CLASSES, conf_threshold = 0.6): # prediction est de la forme (num_anchors, S, S, 5 + num_classes)
	A = prediction.size(0)
	device = prediction.device
	
	# Extract objectness score (confidence)
	objectness = torch.sigmoid(prediction[..., 0])  # Shape: (A, grid_width, grid_height) ?? OU INVERSE ??
	print("objectness.shape = ", objectness.shape)
	print("objectness max min = ", objectness.max(), objectness.min())
	
	
	# Extract box coordinates
	pred_xy = torch.sigmoid(prediction[..., 1:3])   # Shape: (A, S, S, 2)
	pred_wh = torch.exp(prediction[..., 3:5]) * anchors.view(A, 1, 1, 2)  # Shape: (A, S, S, 2)
	print("pred_xy.shape = ", pred_xy.shape)
	print("pred_wh.shape = ", pred_wh.shape)
	print("pred_xy max min = ", pred_xy.max(), pred_xy.min())
	print("pred_wh max min = ", pred_wh.max(), pred_wh.min())
	
	# Generate grid indices [0, 1, ..., S-1] for X and Y
	grid_j, grid_i = torch.meshgrid(
		torch.arange(grid_width, device=device), 
		torch.arange(grid_height, device=device),
		indexing="xy"  # Align with image coordinates (i=row, j=column)
	)
	# Add anchor and channel dimensions: (1, S, S, 1) → (A, S, S, 1)
	cell_x = grid_j.unsqueeze(0).unsqueeze(-1).expand(A, -1, -1, -1)
	cell_y = grid_i.unsqueeze(0).unsqueeze(-1).expand(A, -1, -1, -1)
	print("cell_x.shape = ", cell_x.shape)
	print("cell_y.shape = ", cell_y.shape)
	print("cell_x max min = ", cell_x.max(), cell_x.min())
	print("cell_y max min = ", cell_y.max(), cell_y.min())
	
	# Calcul de x, y absolus (dans l'image)
	x = (pred_xy[..., 0:1] + cell_x) * (resized_img_width / grid_width)
	y = (pred_xy[..., 1:2] + cell_y) * (resized_img_height / grid_height)
	print("x.shape = ", x.shape)
	print("y.shape = ", y.shape)
	print("x max min = ", x.max(), x.min())
	print("y max min = ", y.max(), y.min())
	
	# w, h sont déjà en pixels (grâce à ancres × exp)
	w = pred_wh[..., 0:1] * (resized_img_width / grid_width)
	h = pred_wh[..., 1:2] * (resized_img_height / grid_height)
	print("w.shape = ", w.shape)
	print("h.shape = ", h.shape)
	print("w max min = ", w.max(), w.min())
	print("h max min = ", h.max(), h.min())
	
	# Get class scores
	class_logits = prediction[..., 5 : 5 + num_classes]  # Shape: (A, S, S, num_classes)
	print("class_logits.shape = ", class_logits.shape)
	print("class_logits max min = ", class_logits.max(), class_logits.min())
	class_probs = torch.softmax(class_logits, dim=-1)
	class_conf, class_idx = torch.max(class_probs, dim=-1, keepdim=True)  # Shape: (A, S, S, 1)
	print("class_conf.shape = ", class_conf.shape)
	print("class_idx.shape = ", class_idx.shape)
	print("class_conf max min = ", class_conf.max(), class_conf.min())
	print("class_idx max min = ", class_idx.max(), class_idx.min())

	# Combine all information and reshape to (A*S*S, 6)
	confidence = objectness.unsqueeze(-1) * class_conf  # (A, S, S, 1)
	boxes = torch.cat([x, y, w, h, confidence, class_idx.float()], dim=-1)  # (A, S, S, 6)
	boxes = boxes.reshape(-1, 6)  # Flatten to (A*S*S, 6)
	print("confidence.shape = ", confidence.shape)
	print("confidence max min = ", confidence.max(), confidence.min())
	print("boxes.shape = ", boxes.shape)
	
	# Filter out boxes with low confidence
	mask = boxes[:, 4] >= conf_threshold
	boxes = boxes[mask]
	return boxes

###################################### PLOT IMAGES WITH BOUNDING BOXES AND CLASS LABELS ######################################

def plot_image(image, boxes, class_labels, save_dir=None, filename="image.png"): # attend des boxes de format [class, confidence, x, y, width, height]
																   # avec x,y,width,height entre 0 et 1 et class entre 0 et 12
	"""
	Function to plot images with bounding boxes and class labels.
	
	Args:
		image: Image as numpy array (H, W, C) in [0, 255]
		boxes: List of bounding boxes [class, confidence, x, y, width, height]
		save_dir: Directory to save the image
		filename: Filename to save the image
	"""
	# Create figure and axes 
	fig, ax = plt.subplots(1, figsize=(12, 8))
	fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
	ax.axis("off")

	# Convert image to uint8 if needed
	if image.dtype != np.uint8:
		image = image.astype(np.uint8)
	
	# Display image
	ax.imshow(image)

	# Get color map
	colour_map = plt.get_cmap("tab20b")
	colors = [colour_map(i) for i in np.linspace(0, 1, len(class_labels))]

	if boxes is not None and len(boxes) > 0:
		# Process each box
		for box in boxes:
			class_pred = int(box[0])
			confidence = box[1]
			
			# Skip if confidence is too low
			if confidence < 0.6:
				continue

			# Convert normalized coordinates to pixel coordinates
			h, w = image.shape[:2]
			x_center = box[2] * w
			y_center = box[3] * h
			width = box[4] * w
			height = box[5] * h
			
			# Calculate rectangle boundaries with clamping
			x1 = max(0, min(x_center - width/2, w - 1))
			y1 = max(0, min(y_center - height/2, h - 1))
			x2 = max(0, min(x_center + width/2, w - 1))
			y2 = max(0, min(y_center + height/2, h - 1))
			
			# Skip invalid boxes
			if x2 <= x1 or y2 <= y1:
				continue

			# Create rectangle with visible parameters
			rect = patches.Rectangle(
				(x1, y1),
				x2 - x1,
				y2 - y1,
				linewidth=3,  # Increased line width
				edgecolor=colors[class_pred],
				facecolor="none",
				alpha=0.9  # Add transparency if needed
			)
			ax.add_patch(rect)

			# Add text label with background
			text = f"{class_labels[class_pred]}: {confidence:.2f}"
			text_x = x1
			text_y = max(y1 - 3, 10)  # Keep text within image bounds
			
			ax.text(
				text_x,
				text_y,
				text,
				color="white",
				fontsize=12,
				bbox={
					"facecolor": colors[class_pred],
					"alpha": 0.8,
					"pad": 2,
					"edgecolor": "none"
				}
			)
	else:
		ax.text(10, 20, "No detection", color='yellow', fontsize=12,
                bbox=dict(facecolor='black', alpha=0.5))
	# Save or display
	if save_dir:
		os.makedirs(save_dir, exist_ok=True)
		plt.savefig(
			os.path.join(save_dir, filename),
			bbox_inches="tight",
			pad_inches=0,
			dpi=100
		)
		plt.close()
	else:
		plt.show()
	plt.close()

In [4]:
######################################## SAVE CHECKPOINTS ########################################
# Function to save checkpoint 
def save_checkpoint(model, optimizer, filename="my_checkpoint.pth.tar"): 
	print("==> Saving checkpoint") 
	checkpoint = { 
		"state_dict": model.state_dict(), 
		"optimizer": optimizer.state_dict(), 
	} 
	torch.save(checkpoint, filename)

######################################## LOAD CHECKPOINTS ########################################
# Function to load checkpoint 
def load_checkpoint(checkpoint_file, model, optimizer, lr): 
	print("==> Loading checkpoint") 
	checkpoint = torch.load(checkpoint_file, map_location=DEVICE) 
	model.load_state_dict(checkpoint["state_dict"]) 
	optimizer.load_state_dict(checkpoint["optimizer"]) 

	for param_group in optimizer.param_groups: 
		param_group["lr"] = lr 

In [5]:
############################ DATASET and DATA AUGMENTATION ############################ 
class YoloCompatibleTransform:
	def __init__(self, resized_img_width = RESIZED_IMG_WIDTH, resized_img_height = RESIZED_IMG_HEIGHT,
			     train=True, max_rotation=10, max_translation=0.05,
				 brightness=0.3, contrast=0.3, saturation=0.1, hue=0.01): # or 0.2 0.2 0.2 0.1
		"""
		Transformations compatibles avec YOLO incluant rotations, translations et ColorJitter.
		Préserve l'alignement des boîtes avec les axes horizontal/vertical.
		
		Args:
			image_size (int): Taille finale de l'image (carrée)
			train (bool): Si True, applique les augmentations aléatoires
			max_rotation (float): Angle maximum de rotation en degrés (±)
			max_translation (float): Translation maximum en pourcentage de l'image (±)
			brightness (float): Variation maximum de luminosité (0-1)
			contrast (float): Variation maximum de contraste (0-1)
			saturation (float): Variation maximum de saturation (0-1)
			hue (float): Variation maximum de teinte (0-0.5)
		"""
		self.resized_img_width = resized_img_width
		self.resized_img_height = resized_img_height
		self.train = train
		self.max_rotation = max_rotation
		self.max_translation = max_translation
		self.brightness = brightness
		self.contrast = contrast
		self.saturation = saturation
		self.hue = hue
		self.to_tensor = transforms.ToTensor()
		
	def __call__(self, image, bboxes):
		# Convert image to PIL Image if numpy array
		if isinstance(image, np.ndarray):
			image = Image.fromarray(image)
		
		orig_img_width, orig_img_height = image.size
		
		# Initialize transformation parameters
		angle = 0
		translate = (0, 0)
		do_flip = False
		
		# Apply random transformations only during training
		if self.train:
			# Random horizontal flip
			do_flip = torch.rand(1).item() > 0.5
			
			# Random rotation between -max_rotation and +max_rotation degrees
			angle = torch.rand(1).item() * 2 * self.max_rotation - self.max_rotation
			
			# Random translation between -max_translation and +max_translation percent
			tx = (torch.rand(1).item() * 2 - 1) * self.max_translation * self.resized_img_width
			ty = (torch.rand(1).item() * 2 - 1) * self.max_translation * self.resized_img_height
			translate = (tx, ty)
			
			# Apply ColorJitter
			jitter = transforms.ColorJitter(
				brightness=self.brightness,
				contrast=self.contrast,
				saturation=self.saturation,
				hue=self.hue
			)
			image = jitter(image)
		
		# Calculate scaling factor to maintain aspect ratio (not clean but from previous work)
		scale = min(self.resized_img_width / orig_img_width, self.resized_img_height / orig_img_height) # = 0.096
		new_w, new_h = int(orig_img_width * scale), int(orig_img_height * scale) # new_w, new_h = 576, 384
		
		# Create a blank canvas of the target size
		canvas = Image.new('RGB', (self.resized_img_width, self.resized_img_height), (127, 127, 127))
		
		# Resize image (maintains aspect ratio)
		image = image.resize((new_w, new_h))
		
		# Calculate center position for pasting the resized image
		paste_x = (self.resized_img_width - new_w) // 2
		paste_y = (self.resized_img_height - new_h) // 2
		
		# Apply horizontal flip if needed (before rotation/translation)
		if do_flip:
			image = F.hflip(image)
		
		# Apply rotation and translation
		if angle != 0 or translate != (0, 0):
			# Create larger canvas for rotation/translation to avoid cutting the image
			padding = max(new_w, new_h) // 2
			big_canvas = Image.new('RGB', (self.resized_img_width + 2*padding, self.resized_img_height + 2*padding), (127, 127, 127))
			big_canvas.paste(image, (padding + paste_x, padding + paste_y))
			
			# Apply affine transformation (rotation + translation)
			big_canvas = big_canvas.rotate(angle, resample=Image.BILINEAR, expand=False, 
										translate=(translate[0], translate[1]))
			
			# Crop back to original size
			canvas = big_canvas.crop((padding, padding, padding + self.resized_img_width, padding + self.resized_img_height))
		else:
			# If no rotation/translation, just paste the image directly
			canvas.paste(image, (paste_x, paste_y))
		
		# Transform bounding boxes
		transformed_bboxes = []
		
		# Calculate transformation matrix for bounding boxes
		# For simplicity, we'll apply transformations in reverse order:
		# 1. Scale and center (always applied)
		# 2. Flip (if applicable)
		# 3. Rotate and translate (if applicable)
		
		for box in bboxes:
			x_center, y_center, w, h, cls = box
			
			# Original image coordinates
			x1 = (x_center - w/2) * orig_img_width
			y1 = (y_center - h/2) * orig_img_height
			x2 = (x_center + w/2) * orig_img_width
			y2 = (y_center + h/2) * orig_img_height
			
			# Step 1: Scale coordinates to resized image
			x1 = x1 * scale
			y1 = y1 * scale
			x2 = x2 * scale
			y2 = y2 * scale
			
			# Step 2: Add padding to center in the image
			x1 += paste_x
			y1 += paste_y
			x2 += paste_x
			y2 += paste_y
			
			# Step 3: Apply horizontal flip if needed
			if do_flip:
				x1, x2 = self.resized_img_width - x2, self.resized_img_width - x1
			
			# Step 4: Apply rotation and translation
			if angle != 0 or translate != (0, 0):
				# Convert box to 4 corner points
				points = [
					(x1, y1),  # top-left
					(x2, y1),  # top-right
					(x2, y2),  # bottom-right
					(x1, y2),  # bottom-left
				]
				
				# Apply transformation to each point
				transformed_points = []
				for px, py in points:
					# Center for rotation
					cx, cy = self.resized_img_width / 2, self.resized_img_height / 2
					
					# Translate point to origin
					px -= cx
					py -= cy
					
					# Apply rotation (convert angle to radians)
					angle_rad = -angle * np.pi / 180.0  # Negative for clockwise rotation
					px_rot = px * np.cos(angle_rad) - py * np.sin(angle_rad)
					py_rot = px * np.sin(angle_rad) + py * np.cos(angle_rad)
					
					# Translate back and apply translation
					px = px_rot + cx + translate[0]
					py = py_rot + cy + translate[1]
					
					transformed_points.append((px, py))
				
				# Get axis-aligned bounding box (AABB) of transformed points
				# This ensures the box remains aligned with horizontal/vertical axes
				x_coords = [p[0] for p in transformed_points]
				y_coords = [p[1] for p in transformed_points]
				x1, x2 = min(x_coords), max(x_coords)
				y1, y2 = min(y_coords), max(y_coords)
			
			# Ensure coordinates are within image bounds and calculate intersection with image
			x1_orig, y1_orig, x2_orig, y2_orig = x1, y1, x2, y2
			
			# Clip coordinates to image boundaries
			x1 = np.clip(x1, 0, self.resized_img_width)
			y1 = np.clip(y1, 0, self.resized_img_height)
			x2 = np.clip(x2, 0, self.resized_img_width)
			y2 = np.clip(y2, 0, self.resized_img_height)
			
			# Calculate how much of the box is still inside the image (intersection over original box)
			original_area = (x2_orig - x1_orig) * (y2_orig - y1_orig)
			if original_area <= 0:  # Avoid division by zero
				continue
				
			intersection_area = (x2 - x1) * (y2 - y1)
			visibility_ratio = intersection_area / original_area
			
			# If less than 25% of the box is visible, we skip it
			if visibility_ratio < 0.25:
				continue
			
			# Convert back to YOLO format (x_center, y_center, width, height)
			w = (x2 - x1) / self.resized_img_width
			h = (y2 - y1) / self.resized_img_height
			x_center = (x1 + x2) / 2 / self.resized_img_width
			y_center = (y1 + y2) / 2 / self.resized_img_height
			
			# Add transformed box if it's still visible and large enough
			if w > 0.005 and h > 0.005 and visibility_ratio >= 0.25:  # Minimum size threshold
				transformed_bboxes.append([x_center, y_center, w, h, cls])
		
		# Convert to tensor and normalize
		image_tensor = self.to_tensor(canvas)
		image_tensor = (image_tensor - 0.0) / 1.0  # Simple normalization
		
		return {"image": image_tensor, "bboxes": transformed_bboxes}
		
# Create a dataset class to load the images and labels from the folder 
class Dataset(torch.utils.data.Dataset): 
	def __init__( 
		self, df, image_dir, label_dir, anchors, 
		resized_img_width=RESIZED_IMG_WIDTH, resized_img_height=RESIZED_IMG_HEIGHT,
		grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT,
		num_classes=13, transform=None
	): 
		# Read the csv file with image names and labels 
		self.label_list = df #pd.read_csv(csv_file) #df has two columns: image name (LXXXXXXX.jpg) and label name (LXXXXXXX.txt)
		# Image and label directories 
		self.image_dir = image_dir 
		self.label_dir = label_dir 
		# Image size 
		self.resized_img_width = resized_img_width
		self.resized_img_height = resized_img_height
		# Transformations 
		self.transform = transform 
		# Grid sizes for each scale 
		self.grid_width = grid_width # 576 // 32 = 18
		self.grid_height = grid_height # 384 // 32 = 12
		# Anchor boxes # anchors is initially a list of 3 lists, each containing 3 tuples (width, height) normalized between 0 and 1
		self.anchors = torch.tensor(anchors) # anchors is now a tensor of shape (9, 2)
		# Number of anchor boxes 
		self.num_anchors = len(self.anchors) # 9 anchors
		# Number of classes 
		self.num_classes = num_classes 
		# Ignore IoU threshold 
		self.ignore_iou_thresh = 0.5 # MAYBE CHANGE TO 0.7-0.8 ???????????????????????????????????????????

	def __len__(self): 
		return len(self.label_list) 
	
	def __getitem__(self, idx):
		# Getting the label path 
		label_path = os.path.join(self.label_dir, self.label_list.iloc[idx, 1]) #self.label_list.iloc[idx, 1] récupère la valeur de la colonne 1 (donc label_name) à la ligne idx
		# We are applying roll to move class label to the last column 
		# 5 columns: x, y, width, height, class_label 
		bboxes = np.roll(np.loadtxt(fname=label_path, 
						delimiter=" ", ndmin=2), 4, axis=1).tolist() # np.loadtxt() charge le fichier label_path (fichier .txt) contenant les annotations YOLO.
																	 # ndmin=2 garantit que même s’il n’y a qu’une seule ligne, on obtient un tableau 2D.
																	 # np.roll() déplace la colonne de la classe à la fin du tableau.
																	 # .tolist() crée une liste de liste(s) à partir du tableau numpy.
		# Getting the image path 
		img_path = os.path.join(self.image_dir, self.label_list.iloc[idx, 0]) 
		image = np.array(Image.open(img_path).convert("RGB")) 

		# Augmentations 
		if self.transform: 
			augs = self.transform(image=image, bboxes=bboxes) 
			image = augs["image"] # image is a tensor of shape (3, 416, 416)
			bboxes = augs["bboxes"] # bboxes is a list of bounding boxes in the format [x, y, width, height, class_label] with x,y,w,h normalized between 0 and 1

		# Below assumes 3 scale predictions (as paper) and same num of anchors per scale 
		# target : [probabilities, x, y, width, height, class_label] 
		# for now targets is empty
		# attention first dimension is vertical cells = Y-axis and second dimension is horizontal cells = X-axis
		target = torch.zeros((self.num_anchors, self.grid_height, self.grid_width, 6)) # targets is a tensor of shape (num_anchors_per_scale, s_i, s_i, 6) et 6 car [probabilities, x, y, width, height, class_label]
										  # en gros pour chaque échelle on crée une grille dont chaque case contient 6 valeurs (probabilité, x, y, width, height, class_label)
										  # targets shape:  torch.Size([9, 26, 26, 6])

		# Identify anchor box and cell for each bounding box 
		for box in bboxes: 
			# Calculate iou of each bounding box WITH ANCHOR BOXES
			iou_anchors = iou(torch.tensor(box[2:4]), self.anchors, is_pred=False) # iou_anchors is a tensor of shape (9,)
													  
			# Selecting the best anchor box 
			anchor_indices = iou_anchors.argsort(descending=True, dim=0) # anchor_indices is a tensor of shape (9,)
			x, y, width, height, class_label = box # x,y,width,height,class_label are all normalized between 0 and 1

			# Assigning the bounding box to the best matching anchor box
			has_anchor = False
			
			for anchor_idx in anchor_indices:
				# Identifying the cell to which the bounding box belongs
				# ATTENTION : i correspond à y (rows) et j correspond à x (columns)
				i, j = int(self.grid_height * y), int(self.grid_width * x)
				
				# Check if the anchor box is already assigned
				anchor_taken = target[anchor_idx, i, j, 0]
				
				if not anchor_taken and not has_anchor:
					# Set the probability to 1 
					target[anchor_idx, i, j, 0] = 1

					# Calculating the center of the bounding box relative to the cell 
					x_cell, y_cell = self.grid_width * x - j, self.grid_height * y - i  # x_cell, y_cell are the coordinates of the center of the bounding box relative to the cell (0-1)

					# Calculating the width and height of the bounding box relative to the grid 
					width_cell, height_cell = (width * self.grid_width, height * self.grid_height) # width_cell, height_cell are the width and height of the bounding box relative to the grid

					# Identify the box coordinates 
					box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])

					# Assigning the box coordinates to the target 
					target[anchor_idx, i, j, 1:5] = box_coordinates 

					# Assigning the class label to the target 
					target[anchor_idx, i, j, 5] = int(class_label) 

					# Set the anchor box as assigned
					has_anchor = True

				# If the anchor box is already assigned, check if the IoU is greater than the threshold 
				elif not anchor_taken and iou_anchors[anchor_idx] > self.ignore_iou_thresh:
					# Set the probability to -1 to ignore the anchor box 
					target[anchor_idx, i, j, 0] = -1
		
		# Return the image and the target (now just a single tensor)
		return image, target



train_transform = YoloCompatibleTransform(resized_img_width=RESIZED_IMG_WIDTH, resized_img_height=RESIZED_IMG_HEIGHT, train=True)
test_transform = YoloCompatibleTransform(resized_img_width=RESIZED_IMG_WIDTH, resized_img_height=RESIZED_IMG_HEIGHT, train=False)

In [6]:
############################################### MODEL  ##############################################
# Defining CNN Block 
class CNNBlock(nn.Module): 
	def __init__(self, in_channels, out_channels, use_batch_norm=True, **kwargs): 
		super().__init__() 
		self.conv = nn.Conv2d(in_channels, out_channels, bias=not use_batch_norm, **kwargs) 
		self.bn = nn.BatchNorm2d(out_channels) 
		self.activation = nn.LeakyReLU(0.1) 
		self.use_batch_norm = use_batch_norm 

	def forward(self, x): 
		# Applying convolution 
		x = self.conv(x) 
		# Applying BatchNorm and activation if needed 
		if self.use_batch_norm: 
			x = self.bn(x) 
			return self.activation(x) 
		else: 
			return x
# Defining residual block 
class ResidualBlock(nn.Module): 
	def __init__(self, channels, use_residual=True, num_repeats=1): 
		super().__init__() 
		
		# Defining all the layers in a list and adding them based on number of 
		# repeats mentioned in the design 
		res_layers = [] 
		for _ in range(num_repeats): 
			res_layers += [ 
				nn.Sequential( 
					nn.Conv2d(channels, channels // 2, kernel_size=1), 
					nn.BatchNorm2d(channels // 2), 
					nn.LeakyReLU(0.1), 
					nn.Conv2d(channels // 2, channels, kernel_size=3, padding=1), 
					nn.BatchNorm2d(channels), 
					nn.LeakyReLU(0.1) 
				) 
			] 
		self.layers = nn.ModuleList(res_layers) 
		self.use_residual = use_residual 
		self.num_repeats = num_repeats 
	
	# Defining forward pass 
	def forward(self, x): 
		for layer in self.layers: 
			residual = x 
			x = layer(x) 
			if self.use_residual: 
				x = x + residual 
		return x

# Defining scale prediction class 
class ScalePrediction(nn.Module): 
	def __init__(self, in_channels, num_classes, num_anchors): 
		super().__init__() 
		# Defining the layers in the network 
		self.pred = nn.Sequential( 
			nn.Conv2d(in_channels, 2*in_channels, kernel_size=3, padding=1), 
			nn.BatchNorm2d(2*in_channels), 
			nn.LeakyReLU(0.1), 
			nn.Conv2d(2*in_channels, (num_classes + 5) * num_anchors, kernel_size=1), 
		) 
		self.num_classes = num_classes 
		self.num_anchors = num_anchors
	
	# Defining the forward pass and reshaping the output to the desired output 
	# format: (bztch_size, 9, grid_size, grid_size, num_classes + 5) 
	def forward(self, x): 
		B, _, H, W = x.shape
		output = self.pred(x) 
		output = output.view(B, self.num_anchors, self.num_classes + 5, H, W)
		output = output.permute(0, 1, 3, 4, 2) 
		return output


# Class for defining YOLOv3 model 
class YOLOv3(nn.Module): 
	def __init__(self, in_channels=3, num_classes=NUM_CLASSES): 
		super().__init__() 
		self.num_classes = num_classes 
		self.in_channels = in_channels 

		# Layers list for YOLOv3 
		self.model = nn.Sequential(
			CNNBlock(in_channels, 16, kernel_size=3, stride=1, padding=1), 
			CNNBlock(16, 32, kernel_size=3, stride=2, padding=1), 
			ResidualBlock(32, num_repeats=1),

			CNNBlock(32, 64, kernel_size=3, stride=2, padding=1), 
			ResidualBlock(64, num_repeats=2),

			CNNBlock(64, 128, kernel_size=3, stride=2, padding=1), 
			ResidualBlock(128, num_repeats=4),

			CNNBlock(128, 256, kernel_size=3, stride=2, padding=1), 
			ResidualBlock(256, num_repeats=4),
			
			CNNBlock(256, 512, kernel_size=3, stride=2, padding=1), 
			ResidualBlock(512, num_repeats=2),

			CNNBlock(512, 256, kernel_size=1, stride=1, padding=0), 
			CNNBlock(256, 512, kernel_size=3, stride=1, padding=1),
			ResidualBlock(512, use_residual=False, num_repeats=1), 
			CNNBlock(512, 256, kernel_size=1, stride=1, padding=0),

			ScalePrediction(256, num_classes=num_classes, num_anchors=len(ANCHORS))
		) 
	
	# Forward pass for YOLOv3 with route connections and scale predictions 
	def forward(self, x):
		print("input model shape = ", x.shape)
		x = self.model(x)
		print("output model shape = ", x.shape)
		return x
# input.shape = (batch_size, 3, 416, 416)
# output = tensor de forme (batch_size, num_anchors, grid_size, grid_size, num_classes + 5)



######################################### TRAINING THE MODEL #########################################
# Defining YOLO loss class 
class YOLOLoss(nn.Module): 
	def __init__(self): 
		super().__init__() 
		self.mse = nn.MSELoss()                    # pour les erreurs continues (x, y, w, h, objectness)
		self.bce = nn.BCEWithLogitsLoss()          # pour l'objectness = score (avec logits)
		self.cross_entropy = nn.CrossEntropyLoss() # pour la classification (classes)
		self.sigmoid = nn.Sigmoid()                # pour forcer x, y, objectness ∈ [0,1]
		self.lambda_box = 1.0 # weight for box coordinates loss
		self.lambda_object = 3.0 # weight for no object loss
		self.lambda_no_object = 2.0 # weight for object loss
		self.lambda_class = 3.0 # weight for class loss
	
	def forward(self, pred, target, anchors): # target = (batch_size, 9, 26, 26, 6) = (batch_size, num_anchors, grid_size, grid_size, 6), 6 = (objectness, x, y, w, h, class_label), (x,y) relatif to the cell, (w,h) relative to the grid size
											  # pred = (batch_size, 9, 26, 26, 5 + num_classes))
							  				  # anchors sera le scale_anchors entre 0 et 26
		# Identifying which cells in target have objects 
		# and which have no objects
		obj = target[..., 0] == 1 # obj.sum = nombre d'objets dans la batch
		no_obj = target[..., 0] == 0

		# Calculating No object loss 
		# On applique la Binary Cross Entropy entre le score prédit (logit) et le label (0), pour les cellules sans objet.
		# Cela apprend au modèle à ne pas détecter d’objets là où il n’y en a pas.
		no_object_loss = self.bce(pred[..., 0:1][no_obj], target[..., 0:1][no_obj])

		# anchors est entre 0 et 26
		# Reshaping anchors to match predictions 
		anchors = anchors.reshape(1, pred.size(1), 1, 1, 2).to(pred.device) # pred.size(1) = 9 ==> anchors = (1, 9, 1, 1, 2) ce qui est compatible avec pred[..., 3:5].shape
					#             │     │            │  │  │
					#             │     │            │  │  └─ largeur et hauteur (w, h)
					#             │     │            │  └──── broadcasté sur chaque cellule colonne (W)
					#             │     │            └─────── broadcasté sur chaque cellule ligne (H)
					#             │     └──────────────────── num_anchors (ex: 9)
					#             └────────────────────────── batch_size (à broadcast)
		
		# Box prediction confidence
		pred_copy_for_score_loss = pred.clone() # clone pour ne pas modifier l'original
		
		pred_xy_01_for_score_loss = self.sigmoid(pred_copy_for_score_loss[..., 1:3]) # DONC LE MODELE VA PREDIRE x et y tels que APRES SIGMOID ils sont entre 0 et 1 (dans la cellule)
		pred_wh_ratio_for_score_loss = torch.exp(pred_copy_for_score_loss[..., 3:5]) * anchors # DONC LE MODELE VA PREDIRE w tel que quand on fait exp(w) on a le rapport entre la taille de la boîte et la taille de l’ancre prédite.
		box_preds_for_score_loss = torch.cat([pred_xy_01_for_score_loss, pred_wh_ratio_for_score_loss],dim=-1)

		# Calculating intersection over union for prediction and target 
		ious = iou(box_preds_for_score_loss[obj], target[..., 1:5][obj]).detach()
		# Calculating Object loss
		pred_score_for_score_loss = self.sigmoid(pred_copy_for_score_loss[..., 0:1]) # pred[..., 0:1] = score (logit) de l'objet
		
		object_loss = self.mse(pred_score_for_score_loss[obj], ious * target[...,0:1][obj]) #target[..., 0:1] == 1 quand il y a un objet (sinon 0). But : la sortie pred[..., 0] doit être proche de IoU si l’objet est bien prédit.
		# DONC LE MODELE VA PREDIRE le score tel que APRES SIGMOID il ressemble à IoU (intersection over union) entre la boîte prédite et la boîte cible.
		
		# Predicted box coordinates 
		pred_copy_for_box_loss = pred.clone()
		pred_copy_for_box_loss[..., 1:3] = self.sigmoid(pred_copy_for_box_loss[..., 1:3])
		# Target box coordinates
		target_copy_for_box_loss = target.clone()
		target_copy_for_box_loss[..., 3:5] = torch.log(1e-6 + target_copy_for_box_loss[..., 3:5] / anchors)
		# Calculating box coordinate loss 
		box_loss = self.mse(pred_copy_for_box_loss[..., 1:5][obj], target_copy_for_box_loss[..., 1:5][obj]) 

		# DONC LE RESEAU SORT :
		# x, y qui sont la coord dans la cell APRES SIGMOID
		# w, h: Le modèle prédit le logarithme du ratio entre la taille de la boîte et la taille de l'ancre
		# Claculating class loss 
		class_loss = self.cross_entropy((pred[..., 5:][obj]), 
										 target[..., 5][obj].long()) 

		# Total loss 
		return ( 
			self.lambda_box * box_loss 
			+ self.lambda_object * object_loss 
			+ self.lambda_no_object * no_object_loss 
			+ self.lambda_class * class_loss 
		)

In [ ]:
def evaluate(loader, model, loss_fn, scaled_anchors, device):
	model.eval()
	losses = []
	with torch.no_grad():
		for x, y in loader:
			x = x.to(device)
			y = y.to(device)
			with torch.amp.autocast(device):
				outputs = model(x)
				loss = loss_fn(outputs, y, scaled_anchors)
				losses.append(loss.item())
	mean_loss = sum(losses) / len(losses) if losses else 0
	return mean_loss

# Define the train function to train the model 
def training_loop(loader, model, optimizer, loss_fn, scaler, scaled_anchors): 
	# Creating a progress bar 
	progress_bar = tqdm(loader, leave=True) 

	# Initializing a list to store the losses 
	losses = [] 
	# Iterating over the training data 
	for _, (x, y) in enumerate(progress_bar): 
		x = x.to(device) # x is a tensor of shape (batch_size, 3, 416, 416)
		y = y.to(device) # y0 is a tensor of shape (batch_size, 9, 26, 26, 6) (num_anchors, grid_size, grid_size, 6)

		with torch.amp.autocast(device): 
			# Getting the model predictions 
			outputs = model(x) # tensor (batch_size, 9, grid_size, grid_size, num_classes + 5)
			# Calculating the loss at each scale 
			loss = loss_fn(outputs, y, scaled_anchors) # scaled_anchors is a tensor of shape (9, 2) (num_anchors, 2)

		# Add the loss to the list
		losses.append(loss.item()) 

		# Reset gradients 
		optimizer.zero_grad() 

		# Backpropagate the loss 
		scaler.scale(loss).backward() 

		# Optimization step 
		scaler.step(optimizer) 

		# Update the scaler for next iteration 
		scaler.update() 

		# update progress bar with loss 
		mean_loss = sum(losses) / len(losses) 
		progress_bar.set_postfix(loss=mean_loss)



# Creating the model from YOLOv3 class 
model = YOLOv3().to(DEVICE) 
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Nombre de paramètres entraînables : {num_params}")
# Defining the optimizer 
optimizer = optim.Adam(model.parameters(), lr = learning_rate) 

# Defining the loss function 
loss_fn = YOLOLoss() 

# Defining the scaler for mixed precision training 
scaler = torch.amp.GradScaler(DEVICE)


from sklearn.model_selection import train_test_split

# Read full CSV that has 2 columns: image name (LXXXXXXX.JPG) and label name (LXXXXXXX.txt)
full_df = pd.read_csv("./link_images_labels_YOLO_v3.csv")
# Split into train/validation (80/20)
train_df, val_df = train_test_split(full_df, test_size=0.1)

# Create datasets
train_dataset = Dataset(
	df=train_df,
	image_dir="../dataset_project_iapr2025/train/",
	label_dir="./labels/",
	anchors=ANCHORS,
	transform=train_transform
)
val_dataset = Dataset(
	df=val_df,
	image_dir="../dataset_project_iapr2025/train/",  # Same as train
	label_dir="./labels/",
	anchors=ANCHORS,
	transform=test_transform
)

# Create dataloaders
train_loader = torch.utils.data.DataLoader(
	train_dataset,
	batch_size=BATCH_SIZE,
	shuffle=True,
	pin_memory=True,
)
val_loader = torch.utils.data.DataLoader(
	val_dataset,
	batch_size=BATCH_SIZE,
	shuffle=False,
	pin_memory=True,
)



from torchvision.ops import nms as torch_nms

def save_val_images(model, loader, device, anchors, save_dir, epoch, grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT, resized_img_width=RESIZED_IMG_WIDTH, resized_img_height=RESIZED_IMG_HEIGHT):
	model.eval()
	os.makedirs(save_dir, exist_ok=True)
	with torch.no_grad():
		for batch_idx, (x, _) in enumerate(loader):
			x = x.to(device)
			outputs = model(x) # shape (batch_size, num_anchors, grid_size, grid_size, 5 + num_classes)

			for img_idx in range(x.size(0)):
				pred = outputs[img_idx] # prend l'image img_idx dans le batch, pred_shape = (num_anchor, grid_size, grid_size, 5 + num_classes)

				# Convert predictions to boxes [num_anchors*s*s, 6]
				boxes = convert_cell_to_bboxes(pred, anchors, grid_width, grid_height)
				final_boxes = []

				if boxes.size(0) > 0:
				# Split into components
					coords = boxes[:, :4]      # [x_center, y_center, width, height]
					scores = boxes[:, 4]       # confidences
					labels = boxes[:, 5].int() # class indices
			
					# Convert from center to corner coordinates [x1, y1, x2, y2]
					boxes_corners = torch.zeros_like(coords)
					boxes_corners[:, 0] = coords[:, 0] - coords[:, 2]/2
					boxes_corners[:, 1] = coords[:, 1] - coords[:, 3]/2
					boxes_corners[:, 2] = coords[:, 0] + coords[:, 2]/2
					boxes_corners[:, 3] = coords[:, 1] + coords[:, 3]/2
			
					# Apply NMS
					keep = torch_nms(boxes_corners, scores, iou_threshold=0.5)
					boxes_nms = boxes_corners[keep]
					scores_nms = scores[keep]
					labels_nms = labels[keep]
			
					# Convert to [class, conf, x_center, y_center, width, height] format
					final_boxes = []
					for box, score, label in zip(boxes_nms, scores_nms, labels_nms):
						width = box[2] - box[0]
						height = box[3] - box[1]
						x_center = box[0] + width / 2
						y_center = box[1] + height / 2
						final_boxes.append([
						label.item(), 
						score.item(),
						x_center.item() / resized_img_width,  # Normalize
						y_center.item() / resized_img_height,
						width.item() / resized_img_width,
						height.item() / resized_img_height
						])
				
				img = x[img_idx].permute(1, 2, 0).cpu().numpy()
				img = (img * 255).astype(np.uint8)
				# plot_image attend des boxes de format [class, confidence, x, y, width, height] avec conf,x,y,w,h entre 0 et 1, et class entre 0 et 12
				plot_image(img, final_boxes, class_labels, save_dir=save_dir, filename=f"epoch_{epoch}_img_{batch_idx + img_idx}.png")

	
	model.train()

import os
import matplotlib.pyplot as plt
from tqdm import tqdm

if LOAD_MODEL:
	load_checkpoint(CHECKPOINT_FILE, model, optimizer, learning_rate)
# Training the model 
for e in range(1, epochs+1): 
	print("Epoch:", e) 
	training_loop(train_loader, model, optimizer, loss_fn, scaler, scaled_anchors) 

	# Validation
	val_loss = evaluate(val_loader, model, loss_fn, scaled_anchors, DEVICE)
	print(f"Validation Loss: {val_loss:.4f}")
	
	# Save validation images
	save_val_images(model, val_loader, DEVICE, scaled_anchors, save_dir="val_images", epoch=e, grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT, resized_img_width=RESIZED_IMG_WIDTH, resized_img_height=RESIZED_IMG_HEIGHT)

	# Saving the model 
	if SAVE_MODEL: 
		save_checkpoint(model, optimizer, filename=f"checkpoint.pth.tar")


Nombre de paramètres entraînables : 9913090
Epoch: 1


  0%|          | 0/81 [00:00<?, ?it/s]

input model shape =  torch.Size([1, 3, 384, 576])
output model shape =  torch.Size([1, 9, 12, 18, 18])


  2%|▏         | 2/81 [00:05<03:21,  2.55s/it, loss=9.39]

input model shape =  torch.Size([1, 3, 384, 576])
output model shape =  torch.Size([1, 9, 12, 18, 18])


  4%|▎         | 3/81 [00:07<03:03,  2.35s/it, loss=9.49]

... (1,763 outputs left out)

 95%|█████████▌| 77/81 [02:42<00:08,  2.09s/it, loss=3.39]

input model shape =  torch.Size([1, 3, 384, 576])
output model shape =  torch.Size([1, 9, 12, 18, 18])


 96%|█████████▋| 78/81 [02:44<00:06,  2.09s/it, loss=3.38]

input model shape =  torch.Size([1, 3, 384, 576])
output model shape =  torch.Size([1, 9, 12, 18, 18])


In [8]:
##################################### JUST IF WE WANT TO CHECK THE TRANSFORMATIONS #####################################
def save_gt_images(loader, class_labels, save_dir="gt_visualizations", grid_width=GRID_WIDTH, grid_height=GRID_HEIGHT):
	"""
	Saves images with ground truth bounding boxes from a DataLoader
	
	Args:
		loader: PyTorch DataLoader (train or val)
		class_labels: List of class names
		save_dir: Directory to save visualized images
	"""
	os.makedirs(save_dir, exist_ok=True)
	print(f"Saving ground truth images to {save_dir}/")
	# Color map for different classes
	colour_map = plt.get_cmap("tab20b")
	colors = [colour_map(i) for i in np.linspace(0, 1, len(class_labels))]
	
	for batch_idx, (images, targets) in enumerate(tqdm(loader)):
		# Convert images from tensor to numpy and denormalize
		images = images.permute(0, 2, 3, 1).cpu().numpy() # shape (batch_size, channels, height, width) to (batch_size, height, width, channels)
		images = (images * 255).astype(np.uint8)
		
		# Process each image in the batch
		for img_idx in range(images.shape[0]):
			img = images[img_idx]
			fig, ax = plt.subplots(1, figsize=(12, 8))
			ax.imshow(img)
			
			# Get boxes from the unique scale
			all_boxes = []
			target = targets[img_idx]
			anchor_indices = torch.where(target[..., 0] == 1)  # Cells with objects
			
			for anchor, i, j in zip(*anchor_indices):
				# par ex anchor = 0, i = 7, j = 15
				# Get box coordinates : x,y in [0,1] relative to the cell, w,h in [0,26] so relative to the grid size
				x, y, w, h = target[anchor, i, j, 1:5]
				class_id = target[anchor, i, j, 5].int() # in [0, 12]
				
				# Convert from grid coordinates to image coordinates
				#cell_size = 1/26
				
				# Calculate box coordinates in image space
				x_center = (j + x) / grid_width #* cell_size # remet entre 0 et 1 relative to the grid size
				y_center = (i + y) / grid_height #* cell_size
				width = w / grid_width #* cell_size # remet entre 0 et 1 relative to the grid size
				height = h / grid_height #* cell_size

				all_boxes.append([
					class_id.item(),  # class
					1.0,             # confidence (ground truth)
					x_center.item(),
					y_center.item(),
					width.item(),
					height.item()
				])
			
			# Draw all ground truth boxes
			for box in all_boxes: # all_boxes est toutes les boites d'une image normalisées entre 0 et 1
				class_id, conf, x, y, w, h = box
				
				# Skip if confidence is too low (shouldn't happen for GT)
				if conf < 0.5:
					continue
				
				# Calculate coordinates
				x1 = (x - w/2) * img.shape[1] # trouve le coin supérieur gauche de la boite en pixels (416x416)
				y1 = (y - h/2) * img.shape[0]
				width = w * img.shape[1]
				height = h * img.shape[0]
				
				# Create rectangle patch
				rect = patches.Rectangle(
					(x1, y1), width, height,
					linewidth=2,
					edgecolor=colors[int(class_id)],
					facecolor="none"
				)
				ax.add_patch(rect)
				
				# Add label
				plt.text(
					x1, y1,
					s=f"{class_labels[int(class_id)]} {conf:.2f}",
					color="white",
					verticalalignment="top",
					bbox={"color": colors[int(class_id)], "pad": 0},
				)
			
			# Save image
			plt.axis("off")
			plt.savefig(
				os.path.join(save_dir, f"batch_{batch_idx}_img_{img_idx}.png"),
				bbox_inches="tight",
				pad_inches=0,
				dpi=100
			)
			plt.close()

# Usage example
#save_gt_images(
#    loader=train_loader,
#    class_labels=class_labels,
#    save_dir="train_gt_visualizations"
#)
#print(f"Saved ground truth visualizations to train_gt_visualizations/")


Saving ground truth images to train_gt_visualizations/


100%|██████████| 81/81 [03:03<00:00,  2.26s/it]

Saved ground truth visualizations to train_gt_visualizations/
